# Speech Review — Colab Embedded Workspace

Whisper transcription, human review, MFA forced alignment, and editable Praat TextGrid boundaries in one interface. Choose English or Chinese in the page header; research text and filenames remain unchanged.

The embedded application contains no recordings, reviews, credentials, or model weights. Save and export before the runtime ends. Existing Drive notebook copies do not update automatically when GitHub changes.


## 1

Unpack the application. The source is embedded; no additional ZIP is needed.


In [ ]:
import base64,hashlib,io,zipfile,sys
from pathlib import Path

_bundle = base64.b64decode("...")
assert hashlib.sha256(_bundle).hexdigest() == "36e1020e69a2299d78b9de0420ec0087274a61bc1542f5062faae51fe75a9e4a", "Application bundle checksum failed"
APP_ROOT = Path("/content/speech_review_code/4a831d4e879d")
APP_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_bundle)) as archive:
    for item in archive.infolist():
        destination = (APP_ROOT/item.filename).resolve()
        if APP_ROOT.resolve() not in destination.parents:
            raise ValueError("Bundle path escapes destination")
    archive.extractall(APP_ROOT)
if str(APP_ROOT) not in sys.path:sys.path.insert(0,str(APP_ROOT))
print("Application ready: ",APP_ROOT)


## 2

Install Python dependencies. The public notebook uses a separate environment and repairs missing venv/pip support. Installation failures display logs.


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"-m","pip","install","-r",str(APP_ROOT/"requirements.txt")],check=True)
print("Python dependencies installed")


## 3

Choose storage. Enable PERSIST_TO_DRIVE to authorize Drive and persist inputs, reviews, and boundaries. Otherwise export before disconnecting.


In [ ]:
#@title Storage location
PERSIST_TO_DRIVE = False #@param {type:"boolean"}
import os
SESSION_ROOT=Path("/content/speech_review_session")
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT=Path("/content/drive/MyDrive/SpeechReviewColab")
else:
    SAVE_ROOT=SESSION_ROOT
SAVE_ROOT.mkdir(parents=True,exist_ok=True)
SESSION_ROOT.mkdir(parents=True,exist_ok=True)
os.environ["SPEECH_REVIEW_DATA"]=str(SAVE_ROOT/"projects")
os.environ["SPEECH_REVIEW_ALIGNMENT_WORK"]=str(SESSION_ROOT/"alignment_work")
os.environ["SPEECH_REVIEW_PLAYBACK_CACHE"]=str(SESSION_ROOT/"playback_cache")
os.environ["MFA_ROOT_DIR"]=str(SESSION_ROOT/"mfa_models")
os.environ["HF_HOME"]=str(SESSION_ROOT/"whisper_models")
os.environ["PYTHONUTF8"]="1"
print("Reviews and boundaries directory: ",SAVE_ROOT/"projects")
print("Persistent storage: ",PERSIST_TO_DRIVE)


## 4

Install isolated MFA 3.3.10. Mandarin models mandarin_mfa and mandarin_china_mfa download before the first alignment.


In [ ]:
subprocess.run([sys.executable,str(APP_ROOT/"install_mfa.py")],cwd=APP_ROOT,check=True)
print("MFA installed. Mandarin models download before the first alignment.")


## 5

Check GPU availability and prepare CUDA libraries. Original notebook profiles prefer GPU medium for Number/Homo and GPU large-v3 for Non-homo, with CPU medium fallback.


In [ ]:
import shutil,site
GPU_AVAILABLE=False
if shutil.which("nvidia-smi"):
    gpu_check=subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True)
    GPU_AVAILABLE=gpu_check.returncode==0 and bool(gpu_check.stdout.strip())
    if GPU_AVAILABLE:print("GPU：",gpu_check.stdout.strip())
if GPU_AVAILABLE:
    subprocess.run([sys.executable,"-m","pip","install","nvidia-cublas-cu12","nvidia-cudnn-cu12"],check=True)
    cuda_dirs=[]
    for directory in site.getsitepackages():
        cuda_dirs.extend(str(p) for p in Path(directory).glob("nvidia/*/lib") if p.is_dir())
    os.environ["LD_LIBRARY_PATH"]=os.pathsep.join(cuda_dirs+[os.environ.get("LD_LIBRARY_PATH","")])
else:
    print("No GPU available; using CPU.")


## 6

Start the service. The public version downloads official cloudflared and creates a temporary password-protected URL. The embedded version uses the Colab iframe proxy. Rerunning stops only processes owned by this notebook.


In [ ]:
import json,time,urllib.request,urllib.error
from urllib.parse import urlsplit
from google.colab import output
PORT=8765
proxy_url=output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
proxy=urlsplit(proxy_url)
if proxy.scheme!="https" or not proxy.hostname:
    raise RuntimeError("No valid Colab proxy URL. Run this cell inside Colab.")
os.environ["SPEECH_REVIEW_PROXY_ORIGIN"]=f"https://{proxy.netloc}"
if "REVIEW_PROCESS" not in globals() or REVIEW_PROCESS.poll() is not None:
    REVIEW_LOG_PATH=SESSION_ROOT/"server.log"
    REVIEW_LOG=REVIEW_LOG_PATH.open("a",encoding="utf-8")
    REVIEW_PROCESS=subprocess.Popen([sys.executable,"-m","uvicorn","backend.main:app","--host","127.0.0.1","--port",str(PORT)],cwd=APP_ROOT,env=os.environ.copy(),stdout=REVIEW_LOG,stderr=subprocess.STDOUT)
BASE=f"http://127.0.0.1:{PORT}/api"
def api(path,payload=None):
    data=None if payload is None else json.dumps(payload,ensure_ascii=False).encode()
    request=urllib.request.Request(BASE+path,data=data,headers={"Content-Type":"application/json"})
    try:
        with urllib.request.urlopen(request,timeout=90) as response:return json.load(response)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(exc.read().decode("utf-8",errors="replace")) from exc
for attempt in range(60):
    try:
        print("Service ready: ",api("/alignment/status"));break
    except Exception:
        if REVIEW_PROCESS.poll() is not None or attempt==59:
            print(REVIEW_LOG_PATH.read_text(encoding="utf-8",errors="replace")[-6000:])
            raise RuntimeError("Service startup failed. Check the logs above.")
        time.sleep(1)


## 7

Import recordings. In the public version, leave IMPORT_IN_WORKSPACE=True to upload existing WAV/TXT/TextGrid sets in the alignment workspace. Set it to False to upload a ZIP here. Drive paths must refer to folders in this runtime; Windows paths are not accessible. TXT references are optional.


In [ ]:
#@title Import participant recordings
AUDIO_FOLDER = "" #@param {type:"string"}
CONDITION = "unknown" #@param ["unknown","number","homo","nonhomo"]
from colab_support import extract_inputs
if AUDIO_FOLDER.strip():
    candidates=[Path(AUDIO_FOLDER.strip()).expanduser().resolve()]
    if not candidates[0].is_dir():raise ValueError("Audio folder not found. Check the Drive mount and path.")
else:
    from google.colab import files
    uploaded=files.upload()
    names=[name for name in uploaded if name.lower().endswith(".zip")]
    if len(names)!=1:raise ValueError("Select one recording ZIP at a time.")
    name=names[0]
    raw=uploaded[name]
    archive_key=hashlib.sha256(raw).hexdigest()[:16]
    archive_path=SESSION_ROOT/(archive_key+".zip")
    archive_path.write_bytes(raw)
    candidates=extract_inputs(archive_path,SAVE_ROOT/"inputs"/archive_key)
    del uploaded,raw
if not candidates:raise ValueError("No WAV files found.")
if len(candidates)==1:
    imported=api("/import",{"folder":str(candidates[0]),"condition":CONDITION})
    print("Imported: ",imported["participant"],"; recordings: ",len(imported["trials"]))
else:
    print("Multiple participants found. Import these folders individually in the workspace:")
    for folder in candidates:print(folder)


## 8

Open the workspace. The public version provides a URL and generated login credentials, including for iPad access. The local URL is internal to the VM. The embedded version displays an iframe. Choose English or Chinese in the page header.


In [ ]:
output.serve_kernel_port_as_iframe(PORT,path="/?embedded=1&view=transcription&lang=en",height=1100,cache_in_notebook=False)


## 9

Optional: download project results. Set DOWNLOAD_NOW=True after saving. Exports contain saved transcripts and valid automatic/reviewed alignments separately; input recordings and credentials are excluded.


In [ ]:
#@title Download all results
DOWNLOAD_NOW = False #@param {type:"boolean"}
if DOWNLOAD_NOW:
    from google.colab import files
    projects=api("/projects")
    if not projects:raise ValueError("No project imported yet.")
    result_path=SESSION_ROOT/"speech_review_results.zip"
    with zipfile.ZipFile(result_path,"w",zipfile.ZIP_DEFLATED) as archive:
        for project in projects:
            with urllib.request.urlopen(BASE+f"/projects/{project['id']}/export-everything",timeout=120) as response:
                data=response.read()
            archive.writestr(f"{project['id']}/participant_export.zip",data)
    files.download(str(result_path))
else:
    print("Download skipped. Set DOWNLOAD_NOW=True and rerun when needed.")


## 10

Optional: inspect logs or stop the service. Save and export before stopping. This notebook does not use automatic keepalive.


In [ ]:
#@title Logs and shutdown
SHOW_LOG = False #@param {type:"boolean"}
STOP_NOW = False #@param {type:"boolean"}
if SHOW_LOG:print(REVIEW_LOG_PATH.read_text(encoding="utf-8",errors="replace")[-7000:])
if STOP_NOW and "REVIEW_PROCESS" in globals() and REVIEW_PROCESS.poll() is None:
    REVIEW_PROCESS.terminate()
    REVIEW_PROCESS.wait(timeout=30)
    print("Stopped the service started by this notebook.")


## References

References and limitations. Runtime availability depends on your account. Clear outputs before sharing. Local source paths do not transfer; upload recordings or use Drive. Import reviewed TextGrids to transfer manual boundaries. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html) and [MFA documentation](https://montreal-forced-aligner.readthedocs.io/en/v3.3.2/installation.html).
